# Processo Seletivo Analytica 2026 - **Grupo L**

# Requisitos
- Identificar linhas de ônibus “deficientes” em horário de pico
- Propor otimizações dessas linhas via remanejamento da frota
- Identificar critérios de remanejamento.
- EXTRA: Cruzar dados das “piores linhas” com os bairros, buscando os bairros mais afetados

# Preparação de dados

Para esse problema, realizamos uma busca dentre diversos datasets e encontramos os seguintes dados disponíveis. Abaixo, mostrarei uma visualização de algumas tabelas de dados.

# **OBS**:
Para visualizar outras tabelas csv nos códigos abaixo, basta substituir a string dentro de `caminho_dado` pelo caminho do arquivo que se deseja visualizar. Todos os arquivos utilizados estão na pasta `/content/drive/MyDrive/DadosAnalyticaPS2026-GrupoL`. Assim que tiver o e-mail de vocês, eu compartilho ela no drive.

## SMTR — Sistema Municipal de Transportes

Dados operacionais, financeiros e estruturais do sistema de transporte público do Rio de Janeiro. Filtrei alguns dados e, abaixo, segue a visualização de alguns que se mostraram mais relevantes para atender os requisitos.

### dashboard_bilhetagem_implantacao_jae
Monitoramento GPS Descrição: Dados agregados de localização de veículos por modal:

- BRT
- Ônibus (SPPO)
- Vans (STPL)
- VLT




In [2]:

import pandas as pd

caminho_dado = "/content/drive/MyDrive/DadosAnalyticaPS2026-GrupoL/dados brutos/dashboard_bilhetagem_implantacao_jae-20260430T211213Z-3-001/dashboard_bilhetagem_implantacao_jae/gps_agregado_onibus.csv"
df = pd.read_csv(caminho_dado)
display(df.head())
print(df.shape)

,operadora,id_validador,latitude,longitude,data,estado_equipamento,primeiro_datetime_gps,ultimo_datetime_gps,qtde_min_entre_a_prim_e_ultima_transmissao,qtde_min_distintos_houve_transmissao,qtde_registros_gps,qtde_registros_gps_georreferenciados,percentual_registros_gps_georreferenciados,percentual_transmissao_a_cada_min
0,CITY RIO ROTAS TURISTICAS LTDA,B35T020D00202175,-22.837801,-43.284966,2026-04-26,FECHADO,2026-04-26 00:00:01.126531,2026-04-26 00:04:01.197523,5,5,5,5,1.0,1.0
1,CITY RIO ROTAS TURISTICAS LTDA,B35T020D00204935,-22.836493,-43.285712,2026-04-26,FECHADO,2026-04-26 00:00:00.126985,2026-04-26 00:04:00.017346,5,5,5,5,1.0,1.0
2,CITY RIO ROTAS TURISTICAS LTDA,B35T020D00207037,-22.836385,-43.285718,2026-04-26,FECHADO,2026-04-26 00:00:13.989783,2026-04-26 00:04:13.625286,5,5,5,5,1.0,1.0
3,CITY RIO ROTAS TURISTICAS LTDA,B35T020D00201761,-22.837153,-43.284936,2026-04-26,FECHADO,2026-04-26 00:00:31.879626,2026-04-26 00:04:32.048613,5,5,5,5,1.0,1.0
4,CITY RIO ROTAS TURISTICAS LTDA,B35T020D00202807,-22.836716,-43.285729,2026-04-26,FECHADO,2026-04-26 00:00:49.664117,2026-04-26 00:04:49.78502,5,5,5,5,1.0,1.0


(3057, 14)


### gtfs
Base padronizada de transporte contendo:
- routes: linhas
- stops: paradas(id das paradas e localização). 485884 linhas
- trips: viagens (onibus, id das rotas e direção) 556360 linhas
- shapes: geometria das rotas (ponto de inicio/fim e formato). 13946 linhas. **aponta para shape**
- calendar + calendar_dates: operação temporal (obras, dias da semana, dia de inicio e término). 923 linhas e 34225 linhas
- ordem_servico: onibus, horarios de funcionamento, tamanho do trajedo ida/volta, total viagens ida/volta (dado o dia da semana) 113629 linhas
- frequencies:




In [3]:
caminho_dado = "/content/drive/MyDrive/DadosAnalyticaPS2026-GrupoL/dados brutos/gtfs-20260430T211243Z-3-001/gtfs/frequencies.csv"
df = pd.read_csv(caminho_dado)
display(df.iloc[1:120])
print(df.shape)

,feed_version,feed_start_date,feed_end_date,trip_id,start_time,end_time,headway_secs,exact_times,versao_modelo
1,2023-06-01,2023-06-01,2023-06-15,55f4e628-7e93-4183-b766-de2524df1c3b,17:11:00,20:23:00,1440,0,201d79faee763526a030ff998bebea9782efe961
2,2026-02-02,2026-02-02,2026-02-10,ca93c752-a3d8-4ba5-b6b9-a4a92385726c,05:00:00,06:00:00,360,0,f7495a359559836032529b2d93bc4eff1a079265
3,2024-05-15,2024-05-15,2024-06-02,3f6c3eae-c4d8-444c-9838-d0429a28ffd4,07:46:00,08:10:00,720,0,31a6853cb59b7bd67d2f865fa81c6992ac4e1435
4,2025-01-25,2025-01-25,2025-02-09,0310c444-502c-4729-aba9-d055158e9852,17:45:00,18:11:00,1560,0,6f4042d043fc67e3b7822b43f11afe7aab4b22d3
5,2023-12-16,2023-12-16,2023-12-20,42a51dbd-7f02-43f4-8475-bdb0567468ae,13:03:30,13:47:00,435,0,201d79faee763526a030ff998bebea9782efe961
...,...,...,...,...,...,...,...,...,...
115,2024-06-03,2024-06-03,2024-06-04,99d85c09-71e9-40af-9dc6-f326dc118f2c,22:28:00,26:16:00,1140,0,31a6853cb59b7bd67d2f865fa81c6992ac4e1435
116,2026-01-03,2026-01-03,2026-01-18,816a2d43-3606-4720-bca7-f65ec671edc3,06:03:00,06:57:00,1620,0,8df7635a5781e2e43ecc1920667b18f29516caa0
117,2024-12-31,2024-12-31,2025-01-01,a56d1407-71b1-4a99-ba8c-e038b88789a2,15:13:00,15:35:00,1320,0,3e1630ddb15dca783e340faf23f08f68733f4813
118,2023-03-16,2023-03-16,2023-03-31,37edb768-a5a1-44b3-9dc3-b2c9785be9cd,20:51:00,24:27:00,4320,0,201d79faee763526a030ff998bebea9782efe961


(484555, 9)


### br_rj_riodejaneiro_viagem_zirix
**TABELA CENTRAL**

Dados
- Horas de inicio e chegada de cada linha
- **Aponta para o shape id, veículo e número da linha**

Útil para calcular a duração de cada viagem dada a hora, que pode servir para criar um modelo preditivo que alimenta possíveis cálculos de

```
# Isto está formatado como código
```

otimizações.





In [4]:
caminho_dado = "/content/drive/MyDrive/DadosAnalyticaPS2026-GrupoL/dados brutos/br_rj_riodejaneiro_viagem_zirix-20260430T211251Z-3-001/br_rj_riodejaneiro_viagem_zirix/viagem_informada.csv"
df = pd.read_csv(caminho_dado)
display(df.iloc[1:120])
print(df.shape)

,data,datetime_partida,datetime_chegada,datetime_processamento,datetime_captura,id_veiculo,trip_id,route_id,shape_id,servico,sentido,id_viagem,versao,datetime_ultima_atualizacao
1,2024-12-08,2024-12-08 18:23:39,2024-12-08 19:34:38,2024-12-08 19:34:40,2024-12-08 19:40:00,B10003,NaN,186,421,901,Ida,B10003_901_I_421_20241208182339,0797b1e917d2ed2d50489a35abd2b87dcab54b55,2024-12-10 01:34:25.331602
2,2025-09-19,2025-09-19 06:41:53,2025-09-19 08:27:23,2025-09-19 08:27:25,2025-09-19 08:40:00,D13012,NaN,O0383AAA0A,34402958-de24-498e-b913-3f70f5311747,383,Volta,D13012_383_V_34402958-de24-498e-b913-3f70f5311...,06f003ebcace75a8702b05d063933a3ac6edc0f8,2025-09-26 01:32:16.328995
3,2025-06-27,2025-06-27 04:53:31,2025-06-27 05:11:55,2025-06-27 05:11:58,2025-06-27 05:20:00,A48056,NaN,O0202AAA0A,1760,202,Ida,A48056_202_I_1760_20250627045331,477e243683cf4516bd39ad4ee8ff21793b78279c,2025-07-02 01:32:34.960347
4,2025-02-19,2025-02-19 07:15:43,2025-02-19 09:27:43,2025-02-19 20:17:14,2025-02-19 20:30:00,C47638,NaN,O0390AAA0A,5au5,390,Ida,C47638_390_I_5au5_20250219071543,8b2a8320de08cc624e12dbbd49f8e02da750444f,2025-02-28 01:34:03.91567
5,2025-05-14,2025-05-14 16:41:44,2025-05-14 18:18:46,2025-05-14 18:18:53,2025-05-14 18:30:00,A41124,NaN,O0108AAA0A,tu4h,108,Volta,A41124_108_V_tu4h_20250514164144,d349769b0baa95c568e15e5e7172f14bf2a1aeeb,2025-05-17 01:32:41.649249
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,2024-09-09,2024-09-09 06:04:01,2024-09-09 06:57:31,2024-09-09 06:57:33,2024-09-09 07:10:00,B58028,NaN,O0313AAA0A,rp0h,313,Ida,B58028_313_I_rp0h_20240909060401,01e2893bb9f7740e6a8cbef6e3cc41023eb76c8b,2024-09-11 01:32:31.606445
116,2025-06-10,2025-06-10 10:34:37,2025-06-10 11:04:40,2025-06-10 11:04:40,2025-06-10 11:10:00,D12061,NaN,O0850AAA0A,O0850AAA0AIDU01,850,Ida,D12061_850_I_O0850AAA0AIDU01_20250610103437,70ea3e327bf97de86d1ca8ee6dcea82672b36d17,2025-06-16 01:40:32.326528
117,2025-12-19,2025-12-19 17:19:18,2025-12-19 18:17:48,2025-12-19 18:17:51,2025-12-19 18:30:00,D86092,NaN,O0871AAA0A,snbu,871,Ida,D86092_871_I_snbu_20251219171918,8cece5d825148f36f9f3cf05b7179fc5b7fd6302,2025-12-23 01:32:23.658456
118,2025-01-22,2025-01-22 13:15:21,2025-01-22 14:23:51,2025-01-22 14:23:54,2025-01-22 14:30:00,B31120,NaN,149,370,497,Volta,B31120_497_V_370_20250122131521,9cff872313ce314dcd97d4cecb0e3b9b95d3fc2a,2025-01-31 01:35:15.840984


(256725, 14)


### veiculo
Dados dos veículos do sistema:

- Licenciamento: id de veículo, data da última vistoria, tecnologia, id chassi. 222625 linhas.
- solicitação de licenciamento: solicitações realizadas. Acredito que não está no escopo
- registro agente verao: Acredito que não está no escopo
# - Operação diária
- Inspeções
- Condições operacionais

In [5]:
caminho_dado = "/content/drive/MyDrive/DadosAnalyticaPS2026-GrupoL/dados brutos/veiculo-20260430T211305Z-3-001/veiculo/sppo_veiculo_dia.csv"
df = pd.read_csv(caminho_dado)
display(df.iloc[1:120])
print(df.shape)

,data,id_veiculo,indicadores,status,tecnologia,placa,data_licenciamento,data_infracao,datetime_ultima_atualizacao,versao
1,2024-07-29,A50052,"{""indicador_ar_condicionado"":true,""indicador_a...",Licenciado com ar e não autuado,NaN,NaN,NaN,NaN,2024-07-29 00:00:00,389645e94331a6482e15e038430440ff7db39778
2,2023-06-16,C47893,"{""indicador_ar_condicionado"":true,""indicador_a...",Licenciado com ar e não autuado (023.II),NaN,NaN,NaN,NaN,2023-06-16 00:00:00,NaN
3,2023-03-05,C27106,"{""indicador_ar_condicionado"":true,""indicador_a...",Licenciado com ar e autuado (023.II),NaN,NaN,NaN,NaN,2023-03-05 00:00:00,NaN
4,2023-10-15,D13045,"{""indicador_ar_condicionado"":true,""indicador_a...",Licenciado com ar e não autuado,NaN,NaN,NaN,NaN,2025-07-08 19:52:01.168434,432fd8b43c8d00bd72b10115bcf6649a74371543
5,2023-07-20,B32611,"{""indicador_ar_condicionado"":false,""indicador_...",Licenciado sem ar e não autuado,NaN,NaN,NaN,NaN,2023-07-20 00:00:00,NaN
...,...,...,...,...,...,...,...,...,...,...
115,2024-03-06,C30371,"{""indicador_ar_condicionado"":true,""indicador_a...",Licenciado com ar e não autuado,NaN,NaN,NaN,NaN,2025-07-09 15:19:55.514817,21b44fea9b6195ab051abb2902eb454531b44db2
116,2023-03-11,B28577,"{""indicador_ar_condicionado"":true,""indicador_a...",Licenciado com ar e não autuado (023.II),NaN,NaN,NaN,NaN,2023-03-11 00:00:00,NaN
117,2023-02-14,D86144,"{""indicador_ar_condicionado"":true,""indicador_a...",Licenciado com ar e não autuado (023.II),NaN,NaN,NaN,NaN,2023-02-14 00:00:00,NaN
118,2024-07-10,B32716,"{""indicador_ar_condicionado"":true,""indicador_a...",Licenciado com ar e não autuado,NaN,NaN,NaN,NaN,2024-07-10 00:00:00,7f82815acd73838af72ee1ed091739d16d51b92f


(136939, 10)


# Solução proposta

Com base nos dados acima, iremos analisar os dados para aplicar otimizações na quantidade de veículos por linha de ônibus. O objetivo é diminuir o tempo de espera dos passageiros por linha de ônibus.

Utilizaremos o o intervalo `headway_secs` de início de linha delimitado pela prefeitura na tabela `frequencies.csv` do `gtfs` e, com ele, definiremos que o intervalo médio de espera em um ponto é `headway_secs`/2

(Necessário embasar essa conta acima, sei que ela vem de probabilidade se a gente considerar que ass pessoas chegam nos pontos de forma uniforme e aleatória.)

Em adição, criaremos um modelo preditivo que receberá o tempo de viagem descrito na tabela `viagem_informada.csv` do dataset `br_rj_riodejaneiro_viagem_zirix`, em adição da linha, sentido, horário e dia da semana. O modelo deverá ser  capaz de prever a duração da viagem em qualquer horário e dia futuro.

(Ainda não sei se uma regressão polinomial para cada uma das linhas vai servir para descrever esse tempo de deslocamento dado a hora e dia)

Uniremos os dados de tempo de viagem e intervalo médio nos pontos e, com isso, calcularemos a quantidade de ônibus necessários que essa linha deve possuir naquele momento para cumprir o intervalo médio.

Podemos realizar uma análise dos dados fornecidos e indicar o atraso médio por linha e informações similares, demonstrando a necessidade da nossa aplicação.

Se sobrar tempo, podemos melhorar o modelo para realocar ônibus entre as linhas de forma dinâmica durante o dia, para servir de apoio a secretaria de transportes e empresas de ônibus.

(falta adicionar um tempo de repouso na conta)
(falta embasar esse problema - má distribuição de frota de onibus rj)

## Limitações
Para um primeiro momento, vou considerar que o tempo de deslocamento entre os pontos de ônibus é constante e uniforme. Sei que é uma suposição que foge da realidade do problema, mas, se mostra necessária por conta do trabalho computacional na coleta, tratamento e análise exploratória de milhões de linhas de dados de GPS de ônibus, que devem ser cruzados com a localização das paradas para calcular o deslocamento entre um ponto e outro.

Além disso, com os dados de GPS, seria possível otimizar o modelo para estimar o deslocamento entre trechos da rota com todas as linhas que cruzam esses trechos, sem se limitar a uma análise que considera apenas os ônibus da mesma linha.

Com essa análise por trecho, seria possível aprimorar o modelo para evitar o efeito sanfona (quando os ônibus colam um atrás do outro e chegam juntos no mesmo ponto)



